In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from vamos import optimize
from vamos.foundation.quality_indicators import compute_hypervolume
from vamos.engine.tuning import (
    ModelBasedTuner,
    TuningTask,
    Instance,
    EvalContext,
    build_nsgaii_config_space,
    config_from_assignment,
    save_history_csv,
)

plt.style.use("ggplot")
print("Imports OK")

Imports OK


Definimos el espacio de parámetros

In [2]:
config_space = build_nsgaii_config_space()
param_space = config_space.to_param_space()
print(f"Parameters ({len(param_space.params)}):")
for name, p in param_space.params.items():
    print(f"  {name}: {type(p).__name__}", end="")
    if hasattr(p, "low"):
        print(f" [{p.low}, {p.high}]", end="")
    if hasattr(p, "choices"):
        print(f" {list(p.choices)}", end="")
    print()

if param_space.conditions:
    print(f"\nConditional params: {len(param_space.conditions)}")

Parameters (35):
  pop_size: Int [20, 200]
  offspring_ratio: Categorical [0, 0.25, 0.5, 0.75, 1.0]
  selection: Categorical ['tournament', 'random', 'boltzmann', 'ranking', 'sus']
  use_external_archive: Boolean
  initializer: Categorical ['random', 'lhs', 'scatter', 'sobol', 'halton', 'obl']
  crossover: Categorical ['sbx', 'blx_alpha', 'blx_alpha_beta', 'arithmetic', 'whole_arithmetic', 'laplace', 'fuzzy', 'pcx', 'undx', 'simplex']
  crossover_prob: Real [0.6, 1.0]
  mutation: Categorical ['polynomial', 'linked_polynomial', 'non_uniform', 'gaussian', 'uniform_reset', 'cauchy', 'uniform', 'levy_flight', 'power_law']
  mutation_prob_factor: Real [0.25, 3.0]
  mutation_eta: Real [5.0, 40.0]
  repair: Categorical ['clip', 'reflect', 'random', 'round', 'wrap', 'midpoint']
  archive_unbounded: Boolean
  archive_prune_policy: Categorical ['crowding', 'hv', 'mc_hv', 'knn', 'maxmin', 'ref_dirs']
  selection_size: Int [2, 10]
  crossover_eta: Real [5.0, 40.0]
  crossover_alpha: Real [0.0, 1.0

Definición de instancias

In [3]:
# Parámetros globales para los problemas ZCAT
N_VAR = 30
N_OBJ = 2  # Usaremos 2 objetivos para este ejemplo

instances = [
    Instance(name="zcat1", n_var=N_VAR, kwargs={}),
]

# Creamos la lista de las 20 instancias
instancias_zcat = []
for i in range(2, 21):
    instancias_zcat.append(
        Instance(
            name=f"zcat{i}",
            n_var=N_VAR,
            # Aquí podrías añadir parámetros específicos de ZCAT si los hay,
            kwargs={}
        )
    )

Ajuste función de evaluación

In [4]:
# Punto de referencia para el hipervolumen (ajustado a N_OBJ) // Valor máximo de hipervolumen (a partir de aqui descartamos)
REF_POINT = np.array([1.1] * N_OBJ)

def eval_fn(config: dict, ctx: EvalContext) -> float:
    """Evalúa NSGA-II en una instancia ZCAT específica."""
    
    # Aseguramos un pop_size por defecto si no viene en el config
    if "pop_size" not in config:
        config["pop_size"] = 100
        
    cfg = config_from_assignment("nsgaii", config)

    # Ejecución de la optimización pasando los kwargs de la instancia
    result = optimize(
        ctx.instance.name,
        algorithm="nsgaii",
        algorithm_config=cfg,
        max_evaluations=ctx.budget,
        seed=ctx.seed,
        n_var=ctx.instance.n_var,
        engine="numpy",
        **ctx.instance.kwargs  # Pasa n_obj, level, etc.
    )

    if result.F is None or len(result.F) == 0:
        return 0.0
        
    return compute_hypervolume(result.F, REF_POINT)

Configuración de la tarea

In [5]:
from vamos.engine.tuning import ModelBasedTuner, TuningTask, build_nsgaii_config_space

# 1. Obtener el espacio de configuración pre-construido
config_space = build_nsgaii_config_space()
param_space = config_space.to_param_space()


In [6]:
# --- CONFIGURACIÓN ---
DB_URL = "sqlite:///tfm_final.db"
SAMPLERS = [ "gp"]
PROBLEMS = [f"zcat{i}" for i in range(2, 21)]
N_TRIALS = 50
N_SEEDS = 3  # Robustez
N_JOBS = 4  # Usa todos tus núcleo

# --- BUCLE MAESTRO ---
for sampler_name in SAMPLERS:
    for instancia in instancias_zcat:
        
        study_name = f"{sampler_name}_{instancia.name}_s{N_SEEDS}"
        print(f"\n🚀 Iniciando: {study_name}")
        
        # 1. Definir la instancia y la tarea
        instance = [instancia]
        task = TuningTask(
            name="zcat_tuning_task",
            param_space=param_space,
            instances=instance,
            seeds=[0, 1, 2],                # 3 semillas para demostración rápida
            budget_per_run=15000,           # FEs por ejecución de MOEA
            maximize=True,
            aggregator=lambda scores: float(np.mean(scores)),
        )
        
        # 2. Configurar el Tuner con persistencia
        tuner = ModelBasedTuner(
            task=task,
            max_trials=N_TRIALS,
            backend="optuna",
            optuna_sampler=sampler_name,
            optuna_storage_url=DB_URL,
            optuna_study_name=study_name,
            seed=42,
            n_jobs=4,
            optuna_load_if_exists=True,
        )
        
        # 3. Ejecución
        t0 = time.perf_counter()
        try:
            # Aquí pasas tu función de evaluación (eval_fn)
            tuner.run(eval_fn)
            t_total = time.perf_counter() - t0
            print(f"✅ Finalizado {study_name} en {t_total:.1f}s")
        except Exception as e:
            print(f"❌ Error en {study_name}: {e}")



🚀 Iniciando: gp_zcat2_s3


/home/usuario/david/TFM/VAMOS/src/vamos/engine/tuning/_model_backend_utils.py:90: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  return cls(seed=seed)
[I 2026-05-18 06:20:12,537] A new study created in RDB with name: gp_zcat2_s3
[I 2026-05-18 06:20:46,594] Trial 2 finished with value: 0.5726610797807411 and parameters: {'pop_size': 47, 'offspring_ratio': 1.0, 'selection': 'boltzmann', 'use_external_archive': False, 'initializer': 'random', 'crossover': 'blx_alpha', 'crossover_prob': 0.9322236587140837, 'mutation': 'linked_polynomial', 'mutation_prob_factor': 0.9464400108844375, 'mutation_eta': 37.10005950306919, 'repair': 'reflect', 'archive_unbounded': True, 'archive_prune_policy': 'crowding', 'selection_size': 4, 'crossover_eta': 9.215168446095625, 'crossover_alpha': 0.8115514710527084, 'blxab_alpha': 0.6483717745390162, 'blxab_beta': 0.8442269301858594, 'wa_alpha': 0.2552225305093918, 'laplace_a': -0.9012277240225379

✅ Finalizado gp_zcat2_s3 en 4723.9s

🚀 Iniciando: gp_zcat3_s3


[I 2026-05-18 07:39:35,374] Trial 2 finished with value: 0.23367591107233976 and parameters: {'pop_size': 162, 'offspring_ratio': 1.0, 'selection': 'random', 'use_external_archive': False, 'initializer': 'sobol', 'crossover': 'sbx', 'crossover_prob': 0.7642987295702941, 'mutation': 'gaussian', 'mutation_prob_factor': 1.8451208823637268, 'mutation_eta': 18.52042015342652, 'repair': 'random', 'archive_unbounded': False, 'archive_prune_policy': 'maxmin', 'selection_size': 9, 'crossover_eta': 16.291059379784663, 'crossover_alpha': 0.08835326562140189, 'blxab_alpha': 0.770256968811777, 'blxab_beta': 0.9552686488989498, 'wa_alpha': 0.9182274132138087, 'laplace_a': -0.09919434241830793, 'laplace_b': 1.8057504611010653, 'fuzzy_d': 1.1027474152217653, 'pcx_sigma_eta': 0.28342287205567746, 'pcx_sigma_zeta': 0.22896759979709766, 'undx_zeta': 0.6124979853626956, 'undx_eta': 0.7870008670119175, 'simplex_epsilon': 0.8245819672651585, 'nonuniform_perturbation': 0.19068886180428657, 'gaussian_sigma': 

✅ Finalizado gp_zcat3_s3 en 4504.4s

🚀 Iniciando: gp_zcat4_s3


...

✅ Finalizado gp_zcat18_s3 en 3459.4s

🚀 Iniciando: gp_zcat19_s3


[I 2026-05-18 19:48:04,219] Trial 0 finished with value: 0.13761139218094506 and parameters: {'pop_size': 200, 'offspring_ratio': 0.75, 'selection': 'boltzmann', 'use_external_archive': False, 'initializer': 'random', 'crossover': 'whole_arithmetic', 'crossover_prob': 0.8276239094185165, 'mutation': 'cauchy', 'mutation_prob_factor': 1.3995162119219053, 'mutation_eta': 33.071315499521305, 'repair': 'clip', 'archive_unbounded': False, 'archive_prune_policy': 'mc_hv', 'selection_size': 3, 'crossover_eta': 32.032710605216025, 'crossover_alpha': 0.0978456427379173, 'blxab_alpha': 0.8707759630617352, 'blxab_beta': 0.0073395536320991495, 'wa_alpha': 0.9717809636562813, 'laplace_a': 0.4916606594520503, 'laplace_b': 1.7451347173143161, 'fuzzy_d': 0.9085181636947959, 'pcx_sigma_eta': 0.22633497007504413, 'pcx_sigma_zeta': 0.2086471761953475, 'undx_zeta': 0.8391759403405868, 'undx_eta': 0.7963061944866471, 'simplex_epsilon': 0.24884154809150866, 'nonuniform_perturbation': 0.42281889667202754, 'ga

✅ Finalizado gp_zcat19_s3 en 3384.3s

🚀 Iniciando: gp_zcat20_s3


[I 2026-05-18 20:44:36,755] Trial 0 finished with value: 0.044789114553878885 and parameters: {'pop_size': 159, 'offspring_ratio': 1.0, 'selection': 'random', 'use_external_archive': False, 'initializer': 'halton', 'crossover': 'undx', 'crossover_prob': 0.8247874862293525, 'mutation': 'power_law', 'mutation_prob_factor': 0.8964200050472662, 'mutation_eta': 36.03898687204659, 'repair': 'midpoint', 'archive_unbounded': True, 'archive_prune_policy': 'mc_hv', 'selection_size': 8, 'crossover_eta': 5.339614963628918, 'crossover_alpha': 0.7209761739469798, 'blxab_alpha': 0.5667350104972634, 'blxab_beta': 0.6880225107036476, 'wa_alpha': 0.9596241170040315, 'laplace_a': -0.3069424881869456, 'laplace_b': 1.268071451353016, 'fuzzy_d': 1.0169027814920413, 'pcx_sigma_eta': 0.09842119465394278, 'pcx_sigma_zeta': 0.1776775630727127, 'undx_zeta': 0.2446885061141786, 'undx_eta': 0.45544741552389, 'simplex_epsilon': 0.6014260597055935, 'nonuniform_perturbation': 0.47832305716579676, 'gaussian_sigma': 0.

✅ Finalizado gp_zcat20_s3 en 4691.7s
